# Gray-Box AIA72 — Post-hoc Operational Robustness Stress Test

Stress-tests the **frozen fallback-first v3 policy** under detected AIA missingness, contiguous outages, and detected degradation. This is explicitly **post-hoc robustness analysis**, not a new independent validation.

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

EVAL_DIR = Path("/home/abmoses2000/graybox_aia72_supplementary2026_v3_eval_20261005")
CASES = EVAL_DIR / "supplementary2026_policy_cases.csv.gz"
SUMMARY = EVAL_DIR / "summary.json"

V3_DIR = Path("/home/abmoses2000/graybox_aia72_policy_v3_fallback_first_freeze_20261005")
V3_PROTOCOL = V3_DIR / "protocol.json"

ALARM_DIR = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005")
ALARM_THRESHOLDS = ALARM_DIR / "alarm_thresholds.csv"

CUTS = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005/candidate_cutoffs.csv")
MOND = Path("/home/abmoses2000/graybox_aia72_conformal_mondrian_v2_20261005/mondrian_thresholds.csv")

OUT = Path("/home/abmoses2000/graybox_aia72_posthoc_robustness_v1_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

PRIMARY_PCT = 0.90
SEED = 20261005

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond, msg):
    if not cond:
        raise ValueError(msg)

def binary_metrics(y, pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=int(((y==1)&(pred==1)).sum())
    fp=int(((y==0)&(pred==1)).sum())
    fn=int(((y==1)&(pred==0)).sum())
    tn=int(((y==0)&(pred==0)).sum())
    recall=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    prec=tp/(tp+fp) if tp+fp else np.nan
    f1=2*prec*recall/(prec+recall) if np.isfinite(prec) and np.isfinite(recall) and (prec+recall)>0 else np.nan
    tss=recall+spec-1 if np.isfinite(recall) and np.isfinite(spec) else np.nan
    denom=((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
    hss=2*(tp*tn-fn*fp)/denom if denom else np.nan
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,recall=float(recall),specificity=float(spec),
                precision=float(prec),f1=float(f1),tss=float(tss),hss=float(hss))

for p in [CASES,SUMMARY,V3_PROTOCOL,ALARM_THRESHOLDS,CUTS,MOND]:
    require(p.is_file(),f"Missing input: {p}")

summary=json.loads(SUMMARY.read_text())
v3_protocol=json.loads(V3_PROTOCOL.read_text())

require(summary["status"]=="SUPPLEMENTARY_2026_FALLBACK_V3_CLEAN_EVALUATION_COMPLETE",
        "Unexpected source evaluation status")
require(summary["policy_reselection_performed"] is False,"Source evaluation was reselected")
require(summary["threshold_tuning_performed"] is False,"Source evaluation was tuned")
require(v3_protocol["primary_policy_id"]=="fallback_v3_q90","Primary v3 changed")

df=pd.read_csv(CASES,low_memory=False)
alarm=pd.read_csv(ALARM_THRESHOLDS)
cutoffs=pd.read_csv(CUTS)
alarm_map={r.branch:float(r.threshold) for r in alarm.itertuples()}

def cutoff(metric,pct):
    hit=cutoffs.loc[(cutoffs["metric"]==metric)&np.isclose(cutoffs["percentile"].astype(float),pct)]
    require(len(hit)==1,f"Missing cutoff {metric} q{pct}")
    return float(hit.iloc[0]["cutoff"])

required=[
    "label","issue_utc","sharp_probability","fusion_probability",
    "fusion_seed_std","sharp_aia_probability_gap","fusion_entropy",
    "sharp_aia_mondrian_state"
]
missing=[c for c in required if c not in df.columns]
require(not missing,f"Missing required case columns: {missing}")

y=df["label"].astype(int).to_numpy()
sharp_p=df["sharp_probability"].astype(float).to_numpy()
fusion_p=df["fusion_probability"].astype(float).to_numpy()
sharp_ok=np.isfinite(sharp_p)
base_aia_ok=np.isfinite(fusion_p)

fusion_single=np.isin(
    df["sharp_aia_mondrian_state"].astype(str).to_numpy(),
    ["singleton_no_flare","singleton_flare"]
)

fs=cutoff("fusion_seed_std",PRIMARY_PCT)
gap=cutoff("sharp_aia_probability_gap",PRIMARY_PCT)
ent=cutoff("fusion_entropy",PRIMARY_PCT)

base_normal=(
    base_aia_ok &
    sharp_ok &
    fusion_single &
    (df["fusion_seed_std"].to_numpy(float)<=fs) &
    (df["sharp_aia_probability_gap"].to_numpy(float)<=gap) &
    (df["fusion_entropy"].to_numpy(float)<=ent)
)

sharp_pred=(sharp_p>=alarm_map["sharp"]).astype(int)
fusion_pred=(fusion_p>=alarm_map["sharp_aia"]).astype(int)

issue_dt=pd.to_datetime(df["issue_utc"],utc=True,errors="coerce")
require(issue_dt.notna().all(),"issue_utc parse failure")

def evaluate_scenario(name,aia_available_mask,scenario_type,parameter,replicate=0):
    aia_available_mask=np.asarray(aia_available_mask,bool)

    fusion_issue=aia_available_mask & sharp_ok
    fpred=np.full(len(y),-1,dtype=int)
    fpred[fusion_issue]=fusion_pred[fusion_issue]

    sharp_issue=sharp_ok
    spred=np.full(len(y),-1,dtype=int)
    spred[sharp_issue]=sharp_pred[sharp_issue]

    normal=aia_available_mask & base_normal
    degraded=(~normal)&sharp_ok
    abstain=~(normal|degraded)

    vpred=np.full(len(y),-1,dtype=int)
    vpred[normal]=fusion_pred[normal]
    vpred[degraded]=sharp_pred[degraded]

    out=[]
    for system,issued,pred,norm,deg,absn in [
        ("baseline_always_fusion",fusion_issue,fpred,fusion_issue,np.zeros(len(y),bool),~fusion_issue),
        ("baseline_always_sharp",sharp_issue,spred,np.zeros(len(y),bool),sharp_issue,~sharp_issue),
        ("fallback_v3_q90",normal|degraded,vpred,normal,degraded,abstain),
    ]:
        m=binary_metrics(y[issued],pred[issued]) if issued.any() else {
            k:np.nan for k in ["tp","fp","fn","tn","recall","specificity","precision","f1","tss","hss"]
        }
        positives=int((y==1).sum())
        out.append({
            "scenario":name,
            "scenario_type":scenario_type,
            "parameter":parameter,
            "replicate":replicate,
            "system":system,
            "requested_cases":len(y),
            "positives":positives,
            "normal_cases":int(norm.sum()),
            "degraded_cases":int(deg.sum()),
            "abstain_cases":int(absn.sum()),
            "availability_rate":float(issued.mean()),
            "positive_retention_rate":float(((y==1)&issued).sum()/positives),
            "abstained_flares":int(((y==1)&absn).sum()),
            "false_alarms":int(((y==0)&issued&(pred==1)).sum()),
            **{f"issued_{k}":v for k,v in m.items()}
        })
    return out

rows=[]

rows += evaluate_scenario(
    "clean_reference",
    np.ones(len(df),dtype=bool),
    "reference",
    0.0,
    0
)

rng=np.random.default_rng(SEED)
for rate in [0.05,0.10,0.20,0.40]:
    for rep in range(100):
        mask=rng.random(len(df))>=rate
        rows += evaluate_scenario(
            f"random_missing_{int(rate*100)}pct",
            mask,
            "random_detected_missingness",
            rate,
            rep
        )

unique_times=np.array(sorted(issue_dt.unique()))
cadence_minutes=96
hours_to_steps={6:max(1,round(6*60/cadence_minutes)),
                12:max(1,round(12*60/cadence_minutes)),
                24:max(1,round(24*60/cadence_minutes)),
                48:max(1,round(48*60/cadence_minutes))}

for hours,steps in hours_to_steps.items():
    if steps>=len(unique_times):
        continue
    starts=np.linspace(0,len(unique_times)-steps-1,num=min(50,len(unique_times)-steps),dtype=int)
    starts=np.unique(starts)
    for rep,start in enumerate(starts):
        bad=set(unique_times[start:start+steps])
        mask=~issue_dt.isin(bad).to_numpy()
        rows += evaluate_scenario(
            f"contiguous_outage_{hours}h",
            mask,
            "contiguous_detected_outage",
            float(hours),
            rep
        )

rng=np.random.default_rng(SEED+17)
for rate in [0.05,0.10,0.20,0.40]:
    for rep in range(100):
        healthy=rng.random(len(df))>=rate
        rows += evaluate_scenario(
            f"detected_degradation_{int(rate*100)}pct",
            healthy,
            "detected_aia_degradation",
            rate,
            rep
        )

res=pd.DataFrame(rows)

metrics_to_summarize=[
    "availability_rate","positive_retention_rate","abstained_flares",
    "false_alarms","issued_recall","issued_specificity","issued_precision",
    "issued_f1","issued_tss","issued_hss"
]

agg_rows=[]
for (stype,param,system),g in res.groupby(["scenario_type","parameter","system"],dropna=False):
    row={"scenario_type":stype,"parameter":param,"system":system,"n_replicates":len(g)}
    for m in metrics_to_summarize:
        vals=pd.to_numeric(g[m],errors="coerce").to_numpy(float)
        row[f"{m}_mean"]=float(np.nanmean(vals))
        row[f"{m}_p05"]=float(np.nanpercentile(vals,5))
        row[f"{m}_p95"]=float(np.nanpercentile(vals,95))
    agg_rows.append(row)

agg=pd.DataFrame(agg_rows)

v3=res.loc[res["system"]=="fallback_v3_q90"].copy()
fus=res.loc[res["system"]=="baseline_always_fusion"].copy()
key=["scenario","scenario_type","parameter","replicate"]
paired=v3.merge(fus,on=key,suffixes=("_v3","_fusion"))

delta=pd.DataFrame({
    **{k:paired[k] for k in key},
    "availability_gain_v3_vs_fusion":paired["availability_rate_v3"]-paired["availability_rate_fusion"],
    "flare_retention_gain_v3_vs_fusion":paired["positive_retention_rate_v3"]-paired["positive_retention_rate_fusion"],
    "abstained_flare_reduction_v3_vs_fusion":paired["abstained_flares_fusion"]-paired["abstained_flares_v3"],
    "tss_delta_v3_vs_fusion":paired["issued_tss_v3"]-paired["issued_tss_fusion"],
    "false_alarm_delta_v3_vs_fusion":paired["false_alarms_v3"]-paired["false_alarms_fusion"],
})

res_path=OUT/"scenario_level_results.csv.gz"
agg_path=OUT/"scenario_summary.csv"
delta_path=OUT/"resilience_deltas_v3_vs_fusion.csv"
protocol_path=OUT/"protocol.json"

res.to_csv(res_path,index=False,compression={"method":"gzip","mtime":0})
agg.to_csv(agg_path,index=False)
delta.to_csv(delta_path,index=False)

protocol={
    "status":"POSTHOC_OPERATIONAL_ROBUSTNESS_V1_COMPLETE",
    "scientific_acceptance":False,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "analysis_status":"posthoc_robustness_not_independent_validation",
    "primary_policy_id":"fallback_v3_q90",
    "policy_reselection_performed":False,
    "threshold_tuning_performed":False,
    "policy_redesign_performed":False,
    "source_dataset":"supplementary_2026 already-opened clean evaluation cases",
    "scenarios":{
        "random_detected_missingness_rates":[0.05,0.10,0.20,0.40],
        "random_replicates_per_rate":100,
        "contiguous_detected_outage_hours":[6,12,24,48],
        "contiguous_sampling":"up to 50 deterministic start windows per duration",
        "detected_degradation_rates":[0.05,0.10,0.20,0.40],
        "degradation_interpretation":"AIA health flag marks affected cases unusable; v3 routes them to SHARP fallback."
    },
    "important_limitation":(
        "This notebook evaluates detected outages/degradation. It does not claim to solve silent AIA corruption "
        "or guarantee automatic detection of corrupted imagery."
    ),
    "source_sha256":{
        "supplementary2026_policy_cases":sha256(CASES),
        "supplementary2026_summary":sha256(SUMMARY),
        "v3_protocol":sha256(V3_PROTOCOL),
        "alarm_thresholds":sha256(ALARM_THRESHOLDS),
        "candidate_cutoffs":sha256(CUTS),
        "mondrian_thresholds":sha256(MOND),
    },
    "output_sha256":{
        res_path.name:sha256(res_path),
        agg_path.name:sha256(agg_path),
        delta_path.name:sha256(delta_path),
    }
}
protocol_path.write_text(json.dumps(protocol,indent=2)+"\n")

print("===== POST-HOC OPERATIONAL ROBUSTNESS V1 COMPLETE =====")
print("policy reselection performed: False")
print("threshold tuning performed: False")
print("policy redesign performed: False")
print("status: post-hoc robustness, NOT independent validation")

print("\nReference + scenario summary:")
pd.set_option("display.max_rows",200)
print(agg.to_string(index=False))

print("\nResilience deltas, grouped mean:")
print(
    delta.groupby(["scenario_type","parameter"],dropna=False)
    .agg({
        "availability_gain_v3_vs_fusion":"mean",
        "flare_retention_gain_v3_vs_fusion":"mean",
        "abstained_flare_reduction_v3_vs_fusion":"mean",
        "tss_delta_v3_vs_fusion":"mean",
        "false_alarm_delta_v3_vs_fusion":"mean",
    })
    .reset_index()
    .to_string(index=False)
)

print("\nSaved:",protocol_path)
print("Saved:",res_path)
print("Saved:",agg_path)
print("Saved:",delta_path)


===== POST-HOC OPERATIONAL ROBUSTNESS V1 COMPLETE =====
policy reselection performed: False
threshold tuning performed: False
policy redesign performed: False
status: post-hoc robustness, NOT independent validation

Reference + scenario summary:
              scenario_type  parameter                 system  n_replicates  availability_rate_mean  availability_rate_p05  availability_rate_p95  positive_retention_rate_mean  positive_retention_rate_p05  positive_retention_rate_p95  abstained_flares_mean  abstained_flares_p05  abstained_flares_p95  false_alarms_mean  false_alarms_p05  false_alarms_p95  issued_recall_mean  issued_recall_p05  issued_recall_p95  issued_specificity_mean  issued_specificity_p05  issued_specificity_p95  issued_precision_mean  issued_precision_p05  issued_precision_p95  issued_f1_mean  issued_f1_p05  issued_f1_p95  issued_tss_mean  issued_tss_p05  issued_tss_p95  issued_hss_mean  issued_hss_p05  issued_hss_p95
 contiguous_detected_outage       6.00 baseline_always_f